# Pré-Processamento dos dados

In [12]:
# Importações de depenências

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns


# Carregando o dataset
df = pd.read_excel('../data/raw/churn.xlsx')
pd.options.display.max_columns = None
df.head(2)



,CustomerID,Count,Country,State,City,Zip Code,Lat Long,Latitude,Longitude,Gender,Senior Citizen,Partner,Dependents,Tenure Months,Phone Service,Multiple Lines,Internet Service,Online Security,Online Backup,Device Protection,Tech Support,Streaming TV,Streaming Movies,Contract,Paperless Billing,Payment Method,Monthly Charges,Total Charges,Churn Label,Churn Value,Churn Score,CLTV,Churn Reason
0,3668-QPYBK,1,United States,California,Los Angeles,90003,"33.964131, -118.272783",33.964131,-118.272783,Male,No,No,No,2,Yes,No,DSL,Yes,Yes,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes,1,86,3239,Competitor made better offer
1,9237-HQITU,1,United States,California,Los Angeles,90005,"34.059281, -118.30742",34.059281,-118.307420,Female,No,No,Yes,2,Yes,No,Fiber optic,No,No,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes,1,67,2701,Moved


### Limpeza dos dados

In [13]:
# Primeira limpeza de dados

df = df.drop(['Churn Label', 'Churn Score', 'Churn Reason', 'Country', 'State', 'CustomerID', 'Zip Code', 'Lat Long', 'Latitude', 'Longitude', 'Count'], axis=1)

In [14]:

# Força a conversão para numérico. O que não for número vira NaN.
df['Total Charges'] = pd.to_numeric(df['Total Charges'], errors='coerce')

# (Opcional, mas recomendado) Substituir os NaNs por 0, já que o cliente ainda não pagou nada
df['Total Charges'] = df['Total Charges'].fillna(0)

# Verifique o tipo agora. Deve retornar float64!
print("Tipo da coluna:", df['Total Charges'].dtype)

Tipo da coluna: float64


### Divisão entre treino e teste

In [15]:
# Separar treino e teste
from sklearn.model_selection import train_test_split

y = df['Churn Value']
X = df.drop('Churn Value', axis=1)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

In [16]:
# Colunas numéricas e categóricas
numeric_features = ['Tenure Months', 'Monthly Charges', 'Total Charges', 'CLTV'] 
categorical_features = ['Senior Citizen', 'Partner', 'Dependents', 'Gender', 'Phone Service', 'Multiple Lines', 'Online Security', 'Online Backup', 'Device Protection', 'Tech Support', 'Streaming TV', 'Streaming Movies', 'Contract', 'Paperless Billing', 'Payment Method' ] 
 # Coluna categórica não possui 'City' para evitar maldição da dimensionalidade (muitas cidades diferentes, o que pode prejudicar o modelo)

### Pipeline de pré-processamento

In [18]:
# Importando bibliotecas para pré-processamento
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
    
# 1. Transformador para variáveis numéricas: Apenas normaliza as escalas
numeric_transformer = Pipeline(steps=[
    ('scaler', StandardScaler())
])

# 2. Transformador para variáveis categóricas: Aplica o One-Hot Encoding
categorical_transformer = Pipeline(steps=[
('onehot', OneHotEncoder(handle_unknown='ignore'))
])

# 3. O ColumnTransformer junta tudo:
preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_features),
        ('cat', categorical_transformer, categorical_features)
])

In [19]:
# Aprende a distribuição do Treino e já transforma os dados
X_train_processed = preprocessor.fit_transform(X_train)


# APENAS transforma o Teste usando as regras aprendidas acima (Sem vazamento de dados!)
X_test_processed = preprocessor.transform(X_test)

### Balanceamento de classes

In [20]:
from imblearn.over_sampling import SMOTE

# Cria a instância do SMOTE
smote = SMOTE(random_state=42)

# Aplica APENAS no treino!
X_train_resampled, y_train_resampled = smote.fit_resample(X_train_processed, y_train)
print(f"Tamanho do treino original: {len(y_train)}")
print(f"Tamanho do treino após SMOTE: {len(y_train_resampled)}")



Tamanho do treino original: 5634
Tamanho do treino após SMOTE: 8278


### Salvar artefatos

In [22]:
import joblib
import os

# Cria a pasta caso não exista
os.makedirs('../data/processed', exist_ok=True)

# 1. Salvar os dados prontos para uso
joblib.dump(X_train_resampled, '../data/processed/X_train_resampled.pkl')
joblib.dump(y_train_resampled, '../data/processed/y_train_resampled.pkl')
joblib.dump(X_test_processed, '../data/processed/X_test_processed.pkl')
joblib.dump(y_test, '../data/processed/y_test.pkl')

# 2. Salvar o preprocessor (O "túnel" de transformação) - Super importante para o Deploy depois!
joblib.dump(preprocessor, '../data/processed/preprocessor.pkl')
print("Dados e Pipeline salvos com sucesso na pasta data/processed/!")




Dados e Pipeline salvos com sucesso na pasta data/processed/!
